In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent, ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph, MessagesState
from langchain_core.tools import tool
from typing import Literal

In [2]:
load_dotenv()

True

In [3]:
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI()
parser = StrOutputParser()

Custom Tool

In [4]:
@tool
def get_word_length(word: str) -> int:
    """Returns the length of the word"""
    return len(word)

In [6]:
get_word_length.invoke("Shubham")

7

Predefined Tools

In [9]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
api_wrapper = WikipediaAPIWrapper()
tool = WikipediaQueryRun(api_wrapper=api_wrapper)
print(tool.run({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Retrieval-augmented generation
Summary: Retrieval-augmented generation (RAG) is a technique that enables generative artificial intelligence (Gen AI) models to retrieve and incorporate new information. It modifies interactions with a large language model (LLM) so that the model responds to user queries with reference to a specified set of documents, using this information to supplement information from its pre-existing training data. This allows LLMs to use domain-specific and/or updated information. Use cases include providing chatbot access to internal company data or generating responses based on authoritative sources.
RAG improves la

In [10]:
from langchain_community.tools.tavily_search import TavilySearchResults
search = TavilySearchResults()
search.invoke("what is the weather in SF")

[{'title': 'Monday, April 14, 2025. San Francisco, CA - Weather Forecast',
  'url': 'https://weathershogun.com/weather/usa/ca/san-francisco/480/april/2025-04-14',
  'content': 'San Francisco, California Weather: Monday, April 14, 2025. Cloudy weather, overcast skies with clouds. Day 66°. Night 54°. Precipitation 0 %.',
  'score': 0.94584644},
 {'title': 'Weather in San Francisco',
  'url': 'https://www.weatherapi.com/',
  'content': "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1744669227, 'localtime': '2025-04-14 15:20'}, 'current': {'last_updated_epoch': 1744668900, 'last_updated': '2025-04-14 15:15', 'temp_c': 12.2, 'temp_f': 54.0, 'is_day': 1, 'condition': {'text': 'Mist', 'icon': '//cdn.weatherapi.com/weather/64x64/day/143.png', 'code': 1030}, 'wind_mph': 11.0, 'wind_kph': 17.6, 'wind_degree': 255, 'wind_dir': 'WSW', 'pressure_mb': 1016.0, 'p

Create Agents

In [11]:
from langchain import hub
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

In [13]:
tools = [search]
prompt=hub.pull("hwchase17/openai-functions-agent")
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [15]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [14]:
agent_executor.invoke({"input": "hello how are you?"})



> Entering new AgentExecutor chain...
Hello! I'm here and ready to assist you. How can I help you today?

> Finished chain.


{'input': 'hello how are you?',
 'output': "Hello! I'm here and ready to assist you. How can I help you today?"}

In [16]:
agent_executor.invoke({"input": "What is the current weather in Raleigh, NC?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'current weather in Raleigh, NC'}`


[{'title': 'Monday, April 14, 2025. Raleigh, NC - Weather Forecast', 'url': 'https://weathershogun.com/weather/usa/nc/raleigh/3375/april/2025-04-14', 'content': 'Raleigh, North Carolina Weather: Monday, April 14, 2025. Cloudy weather, overcast skies with clouds. Day 73°. Night 57°. Precipitation 7 %.', 'score': 0.9489978}, {'title': 'Weather in Raleigh, NC', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Raleigh', 'region': 'North Carolina', 'country': 'United States of America', 'lat': 35.7719, 'lon': -78.6389, 'tz_id': 'America/New_York', 'localtime_epoch': 1744669621, 'localtime': '2025-04-14 18:27'}, 'current': {'last_updated_epoch': 1744668900, 'last_updated': '2025-04-14 18:15', 'temp_c': 28.9, 'temp_f': 84.0, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'code

{'input': 'What is the current weather in Raleigh, NC?',
 'output': 'The current weather in Raleigh, NC is partly cloudy with a temperature of 84.0°F. The wind speed is 18.4 kph coming from the WSW direction. The humidity is at 36%, and the visibility is 9.0 miles.'}